# Memory Engineering

An LLM API is **stateless** — the model remembers nothing between calls. Memory is something you *engineer*. This session has two labs:

- **Lab A · Four kinds of memory** — humans use *working*, *episodic*, *semantic*, and *procedural* memory. A chat history is only one of them (episodic). You'll build a runnable example of each.

- **Lab B · When memory runs out** — the context window fills, so you compress older turns into a rolling **summary**. Summarization is **lossy**, so you *test* that the important fact survived — you don't take it on faith.

**Prerequisites:**

✅ basic Python  
✅ a Google (Gemini) API key and  
✅ a Supermemory API key.


> This is a **complete, runnable reference**. Each idea is explained, then implemented; the **`EXPECTED OUTPUT`** blocks show what a cell prints, and the **self-check** (`assert`) cell confirms the summarization test.

### Steps to Obtain API Keys

**1. Google (Gemini AI Studio) API Key**

  * Go to [Google AI Studio](https://aistudio.google.com/).
  * Sign in with your Google account.
  * On the left-hand navigation menu, click on **API keys**.
  * Click the **Create API key** button present at the top right corner.
  * Select an existing Google Cloud project or create a new one, then generate and copy your `GOOGLE_API_KEY`.

**2. Supermemory API Key**

  * Visit [Supermemory.ai](https://supermemory.ai/) and sign in or create an account.
  * Navigate to your dashboard or profile settings.
  * Locate the **API Keys** under _Developer_ section.
  * Click `Create Key` to generate a new API key and copy it for your notebook.

## Setup

Run the next three cells once. They install pinned dependencies, load your credentials, and confirm the environment before you build on it.

In [ ]:
# Pinned for this cohort — do not un-pin, so a framework update can't break the lab.
# (No -q: if an install fails, you want the full pip error visible, not silenced.)
%pip install litellm==1.92.0 supermemory==3.55.0 tiktoken==0.13.0 python-dotenv==1.2.2

print("Dependencies installed. If pip asks you to restart the kernel, do so, then continue.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.1/156.1 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 18.2 MB/s eta 0:00:00
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib_metadata 9.0.0
    Uninstalling importlib_metadata-9.0.0:
      Successfully uninstalled importlib_metadata-9.0.0
Dependencies installed. If pip asks you to restart the kernel, do so, then continue.


In [ ]:
%%writefile .env
GOOGLE_API_KEY=AQ.Ab8RN6Ikk3lu0-3AOtNm5U3EMg1rT7c0NCIISO1nT7Uo929t7Q
SUPERMEMORY_API_KEY=sm_46qwAH7FRfW5B83n5wuhxR_l7zpSZRJu2aYnNCDGC1RTOjq9fW6Cghin4HPYLBQ9Nba6HYpUDpoGNt5BwmVBQCw
LLM_MODEL=gemini/gemini-flash-latest

Writing .env


In [ ]:
# Load credentials. Copy `.env.template` to `.env` and fill in:
#     GOOGLE_API_KEY        ->  required: the LLM, via LiteLLM -> Gemini (summarization + answers)
#     SUPERMEMORY_API_KEY   ->  required: the long-term memory store (get one at console.supermemory.ai)
import os
from dotenv import load_dotenv
load_dotenv(".env", override=True)

# LiteLLM's Gemini provider reads GEMINI_API_KEY; mirror the Google key so one key powers the LLM.
os.environ.setdefault("GEMINI_API_KEY", os.getenv("GOOGLE_API_KEY", ""))

LLM_MODEL = os.getenv("LLM_MODEL", "gemini/gemini-flash-latest")   # stable alias -> current Gemini Flash
# A container tag namespaces a user's memories inside Supermemory (multi-tenant isolation).
USER_ID = "applied-ai-demo-user"

print("Google key      :", bool(os.getenv("GOOGLE_API_KEY")), "(required for the LLM)")
print("Supermemory key :", bool(os.getenv("SUPERMEMORY_API_KEY")), "(required for the memory store)")
print("LLM model       :", LLM_MODEL)
print("Memory namespace:", USER_ID)

Google key      : True (required for the LLM)
Supermemory key : True (required for the memory store)
LLM model       : gemini/gemini-flash-latest
Memory namespace: applied-ai-demo-user


In [ ]:
# Readiness check: confirms libraries import and keys are present. Calls no paid API.
status = {}
for name, module in [("litellm", "litellm"), ("supermemory", "supermemory"), ("tiktoken", "tiktoken")]:
    try:
        __import__(module); status[name] = "ok"
    except Exception as e:
        status[name] = f"import failed: {e}"
status["GOOGLE_API_KEY"]      = "ok" if os.getenv("GOOGLE_API_KEY") else "missing (LLM will fail)"
status["SUPERMEMORY_API_KEY"] = "ok" if os.getenv("SUPERMEMORY_API_KEY") else "missing (memory store will fail)"

print("Environment readiness")
print("---------------------")
for k, v in status.items():
    print(f"  {k:>20} : {v}")

Environment readiness
---------------------
               litellm : ok
           supermemory : ok
              tiktoken : ok
        GOOGLE_API_KEY : ok
   SUPERMEMORY_API_KEY : ok


# Lab A · Four Kinds of Memory

Cognitive science (and the **CoALA** taxonomy for language agents) splits memory into four kinds, and an agent needs all four — not just a chat log:

| Kind | The question it answers | Example |
|------|-------------------------|---------|
| **Working** | *What's in front of me right now?* | The current conversation buffer / context window |
| **Episodic** | *What happened, and when?* | "On 1 Aug the user booked flight AI-302" |
| **Semantic** | *What is stably true?* | "The user is a data scientist who prefers aisle seats" |
| **Procedural** | *How do I do this task?* | "To book travel: check budget freeze, use the corporate vendor…" |

A plain chat history is **episodic** memory — one of the four. The other three you build deliberately. In this lab you'll see one runnable example of each. Working memory lives in-process; the other three we persist in a **memory store** (Supermemory).

## Working memory — what the model sees right now

Working memory is the **current conversation buffer**, bounded by the context window. It's transient: when the session ends, it's gone unless you persist it. Here it's just an in-memory list of the turns so far, plus a token count.

In [ ]:
# tiktoken gives an exact token count; the ~4-chars/token estimate is a fallback if it's unavailable.
def token_count(text: str) -> int:
    try:
        import tiktoken
        return len(tiktoken.get_encoding("cl100k_base").encode(text))
    except Exception:
        return max(1, len(text) // 4)

# Working memory = the live conversation buffer for THIS session.
working_memory = [
    {"role": "user",      "content": "I'm planning a work trip to Delhi next week."},
    {"role": "assistant", "content": "Sure — shall I check the travel policy and your seat preference first?"},
    {"role": "user",      "content": "Yes please, and book me an aisle seat."},
]
used = sum(token_count(m["content"]) for m in working_memory)
print(f"Working memory: {len(working_memory)} turns, ~{used} tokens sitting in the context window.")
print("It's transient — end the session and it's gone, unless a fact graduates to long-term memory.")

# EXPECTED OUTPUT
# ---------------
# Working memory: 3 turns, ~40 tokens sitting in the context window.
# It's transient — end the session and it's gone, unless a fact graduates to long-term memory.

Working memory: 3 turns, ~35 tokens sitting in the context window.
It's transient — end the session and it's gone, unless a fact graduates to long-term memory.


## Write the long-term memories

The other three kinds survive across sessions, so we write them to **Supermemory**. Each `add` returns immediately, but Supermemory indexes new memories **asynchronously**, so we pause briefly before searching. The `metadata["type"]` tag is just for our own clarity in the output.

In [ ]:
import supermemory, time
mem = supermemory.Supermemory(api_key=os.getenv("SUPERMEMORY_API_KEY"))

# Write a memory into the store (each is stored as a document, tagged with its kind).
def remember(text, kind, **extra):
    return mem.add(content=text, container_tag=USER_ID, metadata={"type": kind, **extra})

# Recall from the store by meaning. The extracted-MEMORIES layer is the one that returns readable
# text (each hit has `.memory`), so we query it first — it uses the singular container_tag. We only
# fall back to the documents layer (plural container_tags; text lives in `chunks`) if it's empty.
def recall(query, client=None, k=3):
    client = client or mem
    out = []
    for r in client.search.memories(q=query, container_tag=USER_ID, limit=k).results:
        if getattr(r, "memory", None):
            out.append(((r.metadata or {}).get("type", "?"), r.memory, float(r.similarity or 0.0)))
    if not out:
        for r in client.search.documents(q=query, container_tags=[USER_ID], limit=k).results:
            text = " ".join(c.content for c in (r.chunks or []) if c and c.content).strip()
            if text:
                out.append(((r.metadata or {}).get("type", "?"), text, float(r.score or 0.0)))
    return out

# Episodic — a specific, timestamped event ("what happened").
remember("On 2026-08-01, Alex booked flight AI-302 from Bangalore to Delhi, seat 14A, for a work trip.", "episodic", date="2026-08-01")

# Semantic — stable facts / preferences ("what is true about the user").
remember("Alex is a data scientist. Alex prefers an aisle seat and is allergic to peanuts.", "semantic")

# Procedural — how to carry out a recurring task ("how to").
remember("Travel-booking procedure: (1) check the quarterly budget freeze, (2) use the approved corporate vendor, (3) economy class for flights under 6 hours, (4) submit expenses within 30 days.", "procedural")

# The store extracts memories asynchronously — poll until they're searchable (up to ~60s) rather than guessing a fixed delay. (On a re-run this returns instantly, since the memories already exist.)
print("Waiting for the store to extract the memories (usually a few seconds)...")

for _ in range(20):
    if mem.search.memories(q="flight", container_tag=USER_ID, limit=1).results:
        break
    time.sleep(3)

print("Stored 3 long-term memories (episodic, semantic, procedural) under namespace:", USER_ID)

# EXPECTED OUTPUT
# ---------------
# Stored 3 long-term memories (episodic, semantic, procedural) under namespace: applied-ai-demo-user

Waiting for the store to extract the memories (usually a few seconds)...
Stored 3 long-term memories (episodic, semantic, procedural) under namespace: applied-ai-demo-user


### Episodic memory — recall a specific event

Episodic memory answers **_"what happened?"_**.

We ask about a past event in different words than we stored it, and semantic search over the memory store finds it.

In [ ]:
print("Episodic recall — a specific past event:")

for kind, text, score in recall("what flight did I book for the Delhi trip?", k=2):
    print(f"  {score:.3f}  [{kind}]  {text}")

# EXPECTED OUTPUT (top hit is the flight event, even though the query never says 'AI-302')
# ----------------------------------------------------------------------------------------
# Episodic recall — a specific past event:
#   0.6xx  [episodic]  On 2026-08-01, Alex booked flight AI-302 from Bangalore to Delhi, seat 14A, ...

Episodic recall — a specific past event:
  0.733  [episodic]  Alex booked flight AI-302 from Bangalore to Delhi, seat 14A for a work trip.


### Semantic memory — stable facts and preferences

Semantic memory answers **_"what is stably true?"_**.

We recall the durable facts about the user.

> Note: Supermemory can also distil stored memories into a queryable user *profile* — `mem.profile(container_tag=USER_ID, include=["static"])` — once its background memory-extraction has run.)

In [ ]:
print("Semantic recall — stable facts / preferences:")
for kind, text, score in recall("what are the user's seating preferences and allergies?", k=2):
    print(f"  [{kind}]  {text}")

# EXPECTED OUTPUT
# ---------------
# Semantic recall — stable facts / preferences:
#   [semantic]  Alex is a data scientist. Alex prefers an aisle seat and is allergic to peanuts.

Semantic recall — stable facts / preferences:
  [semantic]  Alex prefers an aisle seat.
  [semantic]  Alex is allergic to peanuts.


### Procedural memory — how to do the task

Procedural memory answers **_"how do I do this?"_**.

It stores reusable routines, not raw transcripts — so when a similar task comes up, the agent recalls the *steps*.

In [ ]:
print("Procedural recall — how to do the task:")
for kind, text, score in recall("how should I book a work trip?", k=3):
    print(f"  [{kind}]  {text}")

# EXPECTED OUTPUT
# ---------------
# Procedural recall — how to do the task:
#   [procedural]  Travel-booking procedure: (1) check the quarterly budget freeze, (2) use the approved corporate vendor, ...

Procedural recall — how to do the task:
  [episodic]  Alex booked flight AI-302 from Bangalore to Delhi, seat 14A for a work trip.


---
# Lab B · When Memory Runs Out

Working memory is bounded by the **context window**. As a conversation grows, "just resend everything" gets slower, costlier, and eventually overflows — and models get *lost* in very long context ("context rot"). The fix isn't a bigger window; it's a deliberate policy: keep the recent turns verbatim, and **compress older turns into a rolling summary**.

But summarization is **lossy** — a summary that *sounds* complete can silently drop the one detail that mattered. So the rule is: **compress, then test recall — don't assume it.**

In [ ]:
# Compress a list of old turns into a short summary, MERGING with any previous summary so earlier facts are never dropped. Real LLM call (Gemini via LiteLLM); temperature=0 for a reproducible test.
import litellm

RECENT_KEEP  = 4      # turns kept verbatim as working memory
TOKEN_BUDGET = 220    # summarise once the buffer exceeds this many tokens

def summarize_turns(turns, prev_summary=""):
    convo = "\n".join(f'{t["role"]}: {t["content"]}' for t in turns)
    system = ("You maintain a running summary of a conversation. Update the existing summary with the "
              "new turns, preserving concrete facts (names, numbers, preferences). Return only the "
              "updated summary, under 100 words.")
    user = f"Existing summary:\n{prev_summary or '(none)'}\n\nNew turns:\n{convo}\n\nUpdated summary:"
    resp = litellm.completion(model=LLM_MODEL, temperature=0, messages=[
        {"role": "system", "content": system}, {"role": "user", "content": user}])
    return resp.choices[0].message.content.strip()

print("summarize_turns ready (LLM:", LLM_MODEL + ").")

summarize_turns ready (LLM: gemini/gemini-flash-latest).


In [ ]:
# Self-check — overflow the budget on purpose, then verify recall. This is the whole point: force a summary, then confirm an early fact survived the lossy compression.
history = []

# 1. Plant a fact in the very first turn.
history.append({"role": "user", "content": "My name is Alex and I have a pet parrot named Kiwi."})
history.append({"role": "assistant", "content": "Nice to meet you, Alex!"})

# 2. Bury it under many long turns to blow past TOKEN_BUDGET.
for i in range(8):
    history.append({"role": "user", "content": f"Explain how volcanoes form, part {i}."})
    history.append({"role": "assistant", "content": "Detailed explanation. " * 12})

# 3. Compress the OLD turns (everything except the last RECENT_KEEP) into a rolling summary.
assert sum(token_count(m["content"]) for m in history) > TOKEN_BUDGET, "budget was not exceeded"
old = history[:-RECENT_KEEP]
summary = summarize_turns(old)

assert "kiwi" in summary.lower(), "Early fact 'Kiwi' was lost — the summary must preserve it."
print("PASS — summarisation fired and the planted fact 'Kiwi' survived the compression.")
print("Rolling summary:", summary)

# EXPECTED OUTPUT (the summary is a real LLM summary, so wording will vary — but 'Kiwi' must appear)
# ------------------------------------------------------------------------------------------------
# PASS — summarisation fired and the planted fact 'Kiwi' survived the compression.
# Rolling summary: Alex introduced themselves and mentioned a pet parrot named Kiwi, then asked a
# series of questions about how volcanoes form. ...

PASS — summarisation fired and the planted fact 'Kiwi' survived the compression.
Rolling summary: Alex has a pet parrot named Kiwi. The assistant provided Alex with a multi-part detailed explanation (Parts 0 through 5) on how volcanoes form.


## Project tie-in — Milestone 3: *Persistent memory*

You now have the two ideas the deck promised, as working code:

- **Four kinds of memory.** You mapped a chat buffer onto **working** memory, and built **episodic**, **semantic**, and **procedural** memory alongside it in a real store.
- **Summarization is lossy.** You compressed old turns into a rolling summary and **tested** — didn't assume — that the planted fact survived.

In your project / assignment, this becomes the assistant's memory layer:
- what it holds in the moment,
- what it summarises, and
- what it writes down to remember you next time.